In [ ]:
import scanpy as sc
import numpy as np
import muon as mu

In [ ]:
adata_f = sc.read('./Data/RNA_ADT/neurIPS/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad')
adata_gex = adata_f
adata_adt = sc.AnnData(X = adata_f.obsm['protein_counts'])

In [5]:
sc.pp.normalize_total(adata_gex, target_sum=1e4)
sc.pp.log1p(adata_gex)
sc.pp.highly_variable_genes(adata_gex, n_top_genes=4000, batch_key='batch')
adata_gex = adata_gex[:, adata_gex.var.highly_variable].copy()
mu.prot.pp.clr(adata_adt)
adata_adt.layers['clr']=adata_adt.X.copy()


adata_adt.var_names = ['PROT_'+name for name in adata_adt.var_names]
adata_gex.var_names = ['RNA_'+name for name in adata_gex.var_names]

In [6]:
mdata = mu.MuData({'rna': adata_gex, 'adt': adata_adt})
print(mdata)
mdata.obs['batch'] = mdata['rna'].obs['batch'].astype('category').copy()
mdata.obs['cell_type'] = mdata['rna'].obs['cell_type'].astype('category').copy()

MuData object with n_obs × n_vars = 90261 × 4134
  2 modalities
    rna:	90261 x 4000
      obs:	'cell_type', 'batch'
      var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'log1p', 'hvg'
      obsm:	'protein_counts'
      layers:	'counts'
    adt:	90261 x 134
      layers:	'clr'


In [ ]:
mu.tl.mofa(mdata, groups_label='batch', gpu_mode=True)

In [ ]:
np.save('MOFA_neurips.npy', mdata.obsm['X_mofa'])